In [1]:
import pandas as pd
import numpy as np
import hashlib
from scipy.stats import norm

In [2]:
df = pd.read_csv("../data/user_agg_sample_1250_users.csv")

In [3]:
def split_user(user_id):
    h = hashlib.md5(str(user_id).encode())

    # TODO:
    # md5 해시값을 16진수 문자열로 변환한 뒤 정수형으로 바꾸고,
    # 2로 나눈 나머지를 반환하세요.
    # 이 함수는 사용자를 50:50에 가깝게 두 그룹으로 나누기 위한 버킷팅 함수입니다.
    return int(h.hexdigest(),16) % 2

In [4]:
def split_user_percent(user_id, b_percent=10):
    h = hashlib.md5(str(user_id).encode())

    # TODO:
    # md5 해시값을 정수형으로 변환한 뒤 100으로 나눈 나머지를 구하세요.
    # 이렇게 만든 값이 b_percent보다 작으면 1(test), 아니면 0(control)을 반환하세요.
    # 이 함수는 전체 사용자 중 일부만 test 그룹에 포함시키는 rollout 상황을 구현합니다.
    val = int(h.hexdigest(),16) % 100

    if val < b_percent:
        return 1
    else:
        return 0

In [5]:
df["variant_50pct"] = df["user_id"].apply(split_user)
df["variant_10pct"] = df["user_id"].apply(lambda x: split_user_percent(x, 10))

In [6]:
# TODO:
# 50:50 Traffic Balance Check에 필요한 값을 계산하세요.
# N은 전체 사용자 수, n_B는 variant_50pct가 1인 사용자 수입니다.
# P는 실제 test 비율, P0는 기대 비율 0.5입니다.
N = len(df)
n_B = df['variant_50pct'].sum()
P = n_B / N
P0 = 0.5

# TODO:
# 50:50 분배 검정을 위한 표준오차(SE), Z-score, p-value를 계산하세요.
# 표준오차는 기대 비율 P0를 기준으로 계산해야 합니다.
SE = np.sqrt(P * (1-P0) / N)
Z = (P-P0) / SE
p_value = 2 * (1 - norm.cdf(abs(Z)))

print("=== 50:50 Traffic Balance Check ===")
print("N:", N)
print("n_B:", n_B)
print("P:", P)
print("SE:", SE)
print("Z:", Z)
print("p-value:", p_value)

if p_value > 0.05:
    print("해석: 50:50 분배가 정상적으로 이루어졌다고 볼 수 있습니다.")
else:
    print("해석: 50:50 분배에 문제가 있을 수 있습니다.")

=== 50:50 Traffic Balance Check ===
N: 1250
n_B: 614
P: 0.4912
SE: 0.014017132374348186
Z: -0.6278031600888827
p-value: 0.5301328957236939
해석: 50:50 분배가 정상적으로 이루어졌다고 볼 수 있습니다.


In [7]:
# TODO:
# 10% rollout 검정도 같은 방식으로 계산하세요.
# 기대 비율만 0.1로 바뀐다는 점에 주의하세요.
N_roll = len(df)
n_B_roll = df['variant_10pct'].sum()
P_roll = n_B_roll / N_roll
P0_roll = 0.1

SE_roll = np.sqrt(P0_roll * (1 - P0_roll) / N_roll)
Z_roll = (P_roll - P0_roll) / SE
p_value_roll = 2 * (1 - norm.cdf(abs(Z_roll)))

print("\n=== 10% Rollout Check ===")
print("N:", N_roll)
print("n_B:", n_B_roll)
print("P:", P_roll)
print("SE:", SE_roll)
print("Z:", Z_roll)
print("p-value:", p_value_roll)

if p_value_roll > 0.05:
    print("해석: 10% rollout이 정상적으로 이루어졌다고 볼 수 있습니다.")
else:
    print("해석: 10% rollout 분배에 문제가 있을 수 있습니다.")


=== 10% Rollout Check ===
N: 1250
n_B: 124
P: 0.0992
SE: 0.00848528137423857
Z: -0.0570730145535356
p-value: 0.9544870326408401
해석: 10% rollout이 정상적으로 이루어졌다고 볼 수 있습니다.


In [8]:
print("\n=== Conversion Rate Z-test ===")

group_A = df[df["variant_50pct"] == 0]
group_B = df[df["variant_50pct"] == 1]

# TODO:
# 각 그룹의 conversion rate를 계산하세요.
# converted 컬럼의 평균을 사용하면 각 그룹의 전환율을 구할 수 있습니다.
conv_A = group_A['converted'].mean()
conv_B = group_B['converted'].mean()

# pooled proportion을 계산합니다.
# 두 그룹의 converted 합을 전체 사용자 수로 나누면 됩니다.
p_pool = (
    group_A["converted"].sum() + group_B["converted"].sum()
) / (len(group_A) + len(group_B))

# TODO:
# pooled proportion 기반 표준오차, Z-score, p-value를 계산하세요.
# conversion rate 비교에서는 각 그룹 크기가 모두 반영되어야 합니다.
se_conv = np.sqrt(p_pool * (1-p_pool) * (1/len(group_A) + 1/len(group_B)))
z_conv = (conv_B - conv_A) / se_conv
p_value_conv = 2 * (1 - norm.cdf(abs(z_conv)))

print("conv_A:", conv_A)
print("conv_B:", conv_B)
print("z:", z_conv)
print("p-value:", p_value_conv)

if p_value_conv > 0.05:
    print("해석: A/B 그룹의 conversion rate 차이는 통계적으로 유의하지 않습니다.")
else:
    print("해석: A/B 그룹의 conversion rate 차이는 통계적으로 유의합니다.")


=== Conversion Rate Z-test ===
conv_A: 0.7955974842767296
conv_B: 0.8045602605863192
z: 0.3960411410672107
p-value: 0.6920746781371256
해석: A/B 그룹의 conversion rate 차이는 통계적으로 유의하지 않습니다.
